# Sakila DVD Rental Data Analysis

## About the Data
This notebook analyzes the **Sakila database**, which contains operational data for a fictional DVD rental business. The data spans across several key business areas: inventory, customer rentals, payments, and stores.

**Role of each CSV file:**
- `actor.csv`: Information about actors.
- `address.csv`: Addresses for stores and customers.
- `category.csv`: Film categories (e.g., Action, Comedy).
- `city.csv`: Cities associated with addresses.
- `country.csv`: Countries associated with cities.
- `customer.csv`: Customer details and account status.
- `film.csv`: Details of every film including title, rating, and rental rate.
- `film_actor.csv`: Links films to actors (which actors are in which films).
- `film_category.csv`: Links films to their categories.
- `inventory.csv`: Physical copies of films stored in specific stores.
- `language.csv`: Languages available for films.
- `payment.csv`: Payment records from customers for rentals.
- `rental.csv`: Records of when specific inventory items were rented and returned by customers.
- `staff.csv`: Staff details.
- `store.csv`: Store locations and manager references.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## 3. Import All CSV Files
We load all the available CSV files into pandas DataFrames using `pd.read_csv()`.

In [ ]:
df_actor = pd.read_csv('actor.csv')
df_address = pd.read_csv('address.csv')
df_category = pd.read_csv('category.csv')
df_city = pd.read_csv('city.csv')
df_country = pd.read_csv('country.csv')
df_customer = pd.read_csv('customer.csv')
df_film = pd.read_csv('film.csv')
df_film_actor = pd.read_csv('film_actor.csv')
df_film_category = pd.read_csv('film_category.csv')
df_inventory = pd.read_csv('inventory.csv')
df_language = pd.read_csv('language.csv')
df_payment = pd.read_csv('payment.csv')
df_rental = pd.read_csv('rental.csv')
df_staff = pd.read_csv('staff.csv')
df_store = pd.read_csv('store.csv')

## 4. Basic Inspection
Let's check the number of rows and columns for each table to understand the scale of the data. We will also display a sample of the most critical table, `payment`.

In [ ]:
tables = {
    'actor': df_actor, 'address': df_address, 'category': df_category,
    'city': df_city, 'country': df_country, 'customer': df_customer,
    'film': df_film, 'film_actor': df_film_actor, 'film_category': df_film_category,
    'inventory': df_inventory, 'language': df_language, 'payment': df_payment,
    'rental': df_rental, 'staff': df_staff, 'store': df_store
}

print("Row counts for imported tables:")
for name, df in tables.items():
    print(f"- {name}: {len(df)} rows")

df_payment.head()

## 5. Data Cleaning
We need to convert datetime strings into proper pandas datetime objects so we can perform time-based analysis. We will clean `payment_date`, `rental_date`, and `return_date`.

In [ ]:
# Convert date columns to datetime objects
df_payment['payment_date'] = pd.to_datetime(df_payment['payment_date'])
df_rental['rental_date'] = pd.to_datetime(df_rental['rental_date'])
df_rental['return_date'] = pd.to_datetime(df_rental['return_date'])

# Check for standard missing values in the rental table (unreturned movies)
print("Missing return dates (active rentals):", df_rental['return_date'].isna().sum())

## 6. Feature Engineering
To better analyze our revenue and operations over time, we will create new columns based on the existing dates.
- `payment_year_month`: Useful for monthly revenue grouping.
- `rental_duration_days`: Useful to understand how long customers keep movies on average.

In [ ]:
# Extract Year and Month from payment date into a formatted string (YYYY-MM)
df_payment['payment_year_month'] = df_payment['payment_date'].dt.to_period('M').astype(str)

# Calculate how many days a rental lasted
df_rental['rental_duration_days'] = (df_rental['return_date'] - df_rental['rental_date']).dt.days

df_payment[['payment_date', 'payment_year_month']].head(3)

## 7. Merges and Master Table
To gain holistic business insights, we need to combine payment, rental, customer, and film information. We do this step by step.

### Merge 1: Payments with Rentals
We merge `df_payment` and `df_rental` using the `rental_id`. This adds rental dates and inventory IDs to every payment record.

### Merge 2: Add Customer Data
We merge the result with `df_customer` using `customer_id`. This adds customer names and emails.

### Merge 3: Add Inventory Data
We merge with `df_inventory` using `inventory_id` to find out which specific film copy was rented.

### Merge 4: Add Film Data
Finally, we merge with `df_film` using `film_id` to get the actual movie titles and rental rates.

In [ ]:
# Merge 1: Payments + Rentals
df_master = pd.merge(df_payment, df_rental[['rental_id', 'rental_date', 'return_date', 'inventory_id', 'rental_duration_days']], on='rental_id', how='left')

# Merge 2: Master + Customers
df_master = pd.merge(df_master, df_customer[['customer_id', 'first_name', 'last_name']], on='customer_id', how='left')

# Merge 3: Master + Inventory
df_master = pd.merge(df_master, df_inventory[['inventory_id', 'film_id']], on='inventory_id', how='left')

# Merge 4: Master + Film
df_master = pd.merge(df_master, df_film[['film_id', 'title', 'rating']], on='film_id', how='left')

df_master.head(2)

## 8. KPI Calculation
Using our newly created `df_master` table, let's calculate standard business KPIs.

In [ ]:
total_revenue = df_payment['amount'].sum()
total_rentals = len(df_rental)
avg_payment = df_payment['amount'].mean()
unreturned_count = df_rental['return_date'].isna().sum()

print("=== Sakila DVD Rental KPIs ===")
print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Total Rentals: {total_rentals}")
print(f"Average Payment Amount: ${avg_payment:,.2f}")
print(f"Rentals Unreturned/Active: {unreturned_count}")

# Top 5 Customers by Revenue
top_customers = df_master.groupby(['first_name', 'last_name'])['amount'].sum().sort_values(ascending=False).head(5)
print("\n--- Top 5 Customers by Revenue ---")
print(top_customers)

## 9. Charts
Visualizing our data helps identify trends and distributions.

### Total Revenue by Month

In [ ]:
monthly_revenue = df_payment.groupby('payment_year_month')['amount'].sum().reset_index()

plt.figure(figsize=(10, 5))
sns.barplot(data=monthly_revenue, x='payment_year_month', y='amount', color='steelblue')
plt.title('Total Revenue over Time')
plt.xlabel('Month')
plt.ylabel('Revenue ($)')
plt.xticks(rotation=45)
plt.show()

**Student Interpretation:**
- We can see total revenue peaked in certain months over the recorded period.
- This bar chart quickly highlights which months were the most profitable for the store.

### Top Movie Ratings Rented

In [ ]:
rating_counts = df_master['rating'].value_counts().reset_index()
rating_counts.columns = ['rating', 'rental_count']

plt.figure(figsize=(8, 5))
sns.barplot(data=rating_counts, x='rating', y='rental_count', palette='viridis')
plt.title('Number of Rentals by Movie Rating')
plt.xlabel('Film Rating')
plt.ylabel('Rental Count')
plt.show()

**Student Interpretation:**
- Knowing the most rented ratings (e.g., PG-13, NC-17) helps management decide what inventory to stock up on.
- Differences in rental popularity represent our core customer demographic priorities.

## 10. Final Summary
- **What the data is about:** A robust dataset encompassing DVD rental transactions, store inventory, films, and customer interactions.
- **Most important result:** Connecting the dots between raw payment transactions and specific film titles reveals complete user journeys and allows deep financial analysis.
- **Key business insights:** Consistently tracking our rental duration and top customers (like identifying our absolute highest spontaneous spenders) can inform future promotional campaigns and loyalty rewards.